# 🧪 PyTorch Lab 3.2: WGAN and WGAN-GP on MNIST

This lab implements two Wasserstein GAN variants on MNIST:

1. **Convolutional WGAN with weight clipping**
2. **Convolutional WGAN-GP with gradient penalty**

Compared with a fully connected implementation, the convolutional version is much more appropriate for images because it exploits spatial locality and local stroke patterns.

**Dataset:** MNIST  
**Main concepts:** GAN, Wasserstein distance, critic, 1-Lipschitz constraint, weight clipping, gradient penalty  
**Recommended duration:** 3 hours



## Practical session format

This version keeps the main structure of the correction notebook, but adds TODO blocks and a few blanks on the parts that are important for understanding GANs and WGANs.

Suggested workflow:
1. Read the theory questions before running the code.
2. Fill the TODOs directly in the notebook.
3. Run short experiments first, then increase the number of epochs.
4. Compare WGAN with weight clipping and WGAN-GP using generated images and training curves.


## 1. From GANs to WGANs

### 1.1 Classical GAN reminder

A classical GAN contains two neural networks:

- a **generator** $G$, which maps random noise $z \sim p(z)$ to a generated image $G(z)$;
- a **discriminator** $D$, which tries to distinguish real images from generated images.

The minimax objective is:

$$
\min_G \max_D 
\mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)]
+
\mathbb{E}_{z \sim p(z)}[\log(1-D(G(z)))].
$$

Classical GANs can be difficult to train because:

- gradients may vanish when the discriminator becomes too strong;
- the Jensen-Shannon divergence can be uninformative when the real and generated distributions have little overlap;
- mode collapse may occur;
- losses are often difficult to interpret.


### 1.2 Wasserstein distance and WGAN

WGAN replaces the classical GAN objective with an approximation of the **Wasserstein-1 distance**, also called **Earth Mover's Distance**.

Intuitively, it measures the minimum cost of transporting probability mass from the generated distribution to the real data distribution.

Using the Kantorovich-Rubinstein dual form:

$$
W(p_{\text{data}}, p_G)
=
\sup_{\|f\|_L \le 1}
\mathbb{E}_{x \sim p_{\text{data}}}[f(x)]
-
\mathbb{E}_{z \sim p(z)}[f(G(z))].
$$

In WGAN:

- the discriminator is replaced by a **critic** $C$;
- the critic outputs a real-valued score, not a probability;
- the critic has **no sigmoid** at the output;
- the critic must be approximately **1-Lipschitz**.

The original WGAN enforces this constraint using **weight clipping**. WGAN-GP replaces clipping with a **gradient penalty**, which is usually more stable.



### TODO 1 - Check the GAN and WGAN objectives

Before coding, answer briefly:

1. In a classical GAN, what is the role of the discriminator?
2. In WGAN, why do we call the discriminator a critic?
3. Why does the critic output a real-valued score instead of a probability?
4. What does the gap $\mathbb{E}[C(x_{\text{real}})] - \mathbb{E}[C(x_{\text{fake}})]$ represent empirically?
5. Why is a 1-Lipschitz constraint needed in WGAN?


In [ ]:
import os
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Subset


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


## 2. Load MNIST

The generator ends with `tanh`, so generated images are in $[-1,1]$.

Therefore, MNIST images are normalized from $[0,1]$ to $[-1,1]$:

$$
x_{\text{norm}} = \frac{x - 0.5}{0.5}.
$$

For local execution, the default configuration uses a subset of MNIST. On a GPU, set `MAX_TRAIN_SAMPLES = None`.


In [ ]:
BATCH_SIZE = 64
MAX_TRAIN_SAMPLES = 20000  # Set to None to use the full MNIST training set.

# num_workers=0 is the safest default for local Jupyter notebooks,
# especially on Windows. Increase it on Linux/macOS if desired.
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

if MAX_TRAIN_SAMPLES is not None:
    indices = list(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
    train_dataset = Subset(train_dataset, indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))


In [ ]:
def denormalize(x):
    """Convert images from [-1, 1] back to [0, 1] for visualization."""
    return ((x + 1) / 2).clamp(0, 1)


def show_images(images, title=None, nrow=8, figsize=(8, 4)):
    images = images.detach().cpu()
    grid = make_grid(denormalize(images), nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()


real_batch, real_labels = next(iter(train_loader))
print("Batch shape:", real_batch.shape)
print("Min value:", real_batch.min().item(), "Max value:", real_batch.max().item())
show_images(real_batch[:32], title="Real MNIST images")



### TODO 2 - Inspect the data normalization

Check the displayed MNIST images and the printed min/max values.

Questions:
1. Why are images normalized to $[-1, 1]$ instead of $[0, 1]$ here?
2. Which activation should the generator use at the output to match this range?
3. What could go wrong if the real images and generated images are not in the same value range?


## 3. Convolutional Generator and Critic

The previous MLP version is too weak for image generation: it does not exploit local spatial structure. Here we use a DCGAN-style architecture.

### Generator

The generator maps:

$$
z \in \mathbb{R}^{d_z} \rightarrow \hat{x} \in \mathbb{R}^{1 \times 28 \times 28}.
$$

It first projects the latent vector to a low-resolution feature map, then upsamples it with transposed convolutions.

### Critic

The critic maps:

$$
x \in \mathbb{R}^{1 \times 28 \times 28} \rightarrow C(x) \in \mathbb{R}.
$$

Important rules:

- no sigmoid at the output;
- no binary cross entropy;
- for WGAN-GP, avoid `BatchNorm` inside the critic because the gradient penalty is computed per sample;
- the generator can use `BatchNorm`.



### TODO 3 - Complete the generator output and critic interpretation

The convolutional architecture is already provided. Fill only the part that is conceptually important for GAN training:

1. The generator output activation should match the image normalization range.
2. The critic must output an unconstrained real-valued score, not a probability.
3. Explain why there is no `Sigmoid` and no `BCELoss` in WGAN.


In [ ]:
LATENT_DIM = 100

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim

        self.project = nn.Sequential(
            nn.Linear(latent_dim, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.ReLU(inplace=True)
        )

        self.net = nn.Sequential(
            nn.Unflatten(1, (256, 7, 7)),

            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 7x7 -> 14x14
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # 14x14 -> 28x28
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            # TODO: choose the activation that matches images normalized in [-1, 1].
            ...
        )

    def forward(self, z):
        x = self.project(z)
        return self.net(x)


class Critic(nn.Module):
    def __init__(self):
        super().__init__()

        # No BatchNorm here. This is important for WGAN-GP.
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),    # 28x28 -> 14x14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 14x14 -> 7x7
            nn.LeakyReLU(0.2, inplace=True),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1)
        )

    def forward(self, x):
        return self.net(x).view(-1)


In [ ]:
def initialize_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(module.weight, mean=1.0, std=0.02)
        nn.init.constant_(module.bias, 0.0)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad_(flag)


generator = Generator(LATENT_DIM).to(device)
critic = Critic().to(device)

generator.apply(initialize_weights)
critic.apply(initialize_weights)

print(generator)
print(critic)
print("Generator parameters:", count_parameters(generator))
print("Critic parameters:", count_parameters(critic))

z = torch.randn(8, LATENT_DIM, device=device)
fake = generator(z)
scores = critic(fake)

print("Generated batch shape:", fake.shape)
print("Critic score shape:", scores.shape)
print("Generated value range:", fake.min().item(), fake.max().item())
assert fake.shape == (8, 1, 28, 28)
assert scores.shape == (8,)


## 4. WGAN losses

The critic approximates:

$$
\mathbb{E}[C(x_{\text{real}})] - \mathbb{E}[C(x_{\text{fake}})].
$$

Because PyTorch optimizers minimize, we use:

$$
L_C =
\mathbb{E}[C(x_{\text{fake}})]
-
\mathbb{E}[C(x_{\text{real}})].
$$

The generator tries to maximize critic scores on fake images:

$$
L_G = - \mathbb{E}[C(x_{\text{fake}})].
$$



### TODO 4 - Fill the WGAN losses

Complete the two losses using the critic scores:

$$
L_C = \mathbb{E}[C(x_{\text{fake}})] - \mathbb{E}[C(x_{\text{real}})]
$$

$$
L_G = -\mathbb{E}[C(x_{\text{fake}})]
$$

Questions:
1. Why is the critic loss written with this sign in PyTorch?
2. Why does the generator minimize the negative critic score on fake images?
3. Why do we not use labels 0 and 1 here?


In [ ]:
def critic_loss_wgan(real_scores, fake_scores):
    # TODO: PyTorch minimizes this loss.
    return ...


def generator_loss_wgan(fake_scores):
    # TODO: the generator wants fake images to receive high critic scores.
    return ...


## 5. Convolutional WGAN with weight clipping

The original WGAN enforces the 1-Lipschitz constraint by clipping all critic weights to a small interval:

$$
w \leftarrow \text{clip}(w, -c, c).
$$


Compared with the previous MLP version, this convolutional version should produce better samples. However, weight clipping is still a crude constraint, so WGAN-GP is usually more stable.

Important implementation choices:

- use `RMSprop` for the original WGAN;
- no `sigmoid`;
- no `BCELoss`;
- train the critic several times before one generator update;
- use `detach()` for fake images during critic training;
- no `detach()` during generator training.

The critic score gap

$$
\mathbb{E}[C(x_{\text{real}})] - \mathbb{E}[C(x_{\text{fake}})]
$$

is an empirical estimate of the Wasserstein distance, not an exact distance.



### TODO 5 - Complete the WGAN training logic

Fill the missing steps that distinguish critic training from generator training. Focus on the GAN logic, not on PyTorch syntax.

Questions:
1. Why is `detach()` used when fake images are passed to the critic during critic training?
2. Why is `detach()` not used during generator training?
3. Why is the critic updated several times before one generator update?
4. What is the purpose of clipping critic weights?
5. How should the empirical Wasserstein estimate evolve if training improves?


In [ ]:
EPOCHS_WGAN = 200
N_CRITIC_WGAN = 5
CLIP_VALUE = 0.05
LR_WGAN = 5e-5

wgan_generator = Generator(LATENT_DIM).to(device)
wgan_critic = Critic().to(device)
wgan_generator.apply(initialize_weights)
wgan_critic.apply(initialize_weights)

optimizer_G = optim.RMSprop(wgan_generator.parameters(), lr=LR_WGAN)
optimizer_C = optim.RMSprop(wgan_critic.parameters(), lr=LR_WGAN)

fixed_noise = torch.randn(32, LATENT_DIM, device=device)

history_wgan = {
    "critic_loss": [],
    "generator_loss": [],
    "wasserstein_estimate": [],
    "real_score": [],
    "fake_score": []
}

critic_updates = 0
start_time = time.time()

for epoch in range(1, EPOCHS_WGAN + 1):
    wgan_generator.train()
    wgan_critic.train()

    epoch_c_loss = []
    epoch_g_loss = []
    epoch_w_est = []
    epoch_real_score = []
    epoch_fake_score = []

    for batch_idx, (real_images, _) in enumerate(train_loader):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # -------------------------
        # 1. Train critic
        # -------------------------
        set_requires_grad(wgan_critic, True)
        set_requires_grad(wgan_generator, False)

        z = torch.randn(batch_size, LATENT_DIM, device=device)
        # TODO: generate fake images for critic training.
        # Important: do not update the generator during this step.
        fake_images = ...

        # TODO: compute critic scores on real and fake images.
        real_scores = ...
        fake_scores = ...

        # TODO: compute the WGAN critic loss.
        c_loss = ...

        optimizer_C.zero_grad(set_to_none=True)
        c_loss.backward()
        optimizer_C.step()

        # Weight clipping for the original WGAN Lipschitz constraint.
        for p in wgan_critic.parameters():
            # TODO: enforce the WGAN Lipschitz constraint by weight clipping.
            p.data.clamp_(..., ...)

        critic_updates += 1

        wasserstein_estimate = real_scores.mean().item() - fake_scores.mean().item()

        epoch_c_loss.append(c_loss.item())
        epoch_w_est.append(wasserstein_estimate)
        epoch_real_score.append(real_scores.mean().item())
        epoch_fake_score.append(fake_scores.mean().item())

        # -------------------------
        # 2. Train generator every N_CRITIC_WGAN critic updates
        # -------------------------
        if critic_updates % N_CRITIC_WGAN == 0:
            set_requires_grad(wgan_critic, False)
            set_requires_grad(wgan_generator, True)

            z = torch.randn(batch_size, LATENT_DIM, device=device)
            # TODO: generate fake images for generator training.
            # Important: gradients must flow into the generator here.
            fake_images = ...
            fake_scores_for_g = ...
            g_loss = ...

            optimizer_G.zero_grad(set_to_none=True)
            g_loss.backward()
            optimizer_G.step()

            epoch_g_loss.append(g_loss.item())

    set_requires_grad(wgan_critic, True)
    set_requires_grad(wgan_generator, True)

    mean_c_loss = float(np.mean(epoch_c_loss))
    mean_g_loss = float(np.mean(epoch_g_loss)) if epoch_g_loss else float("nan")
    mean_w_est = float(np.mean(epoch_w_est))
    mean_real_score = float(np.mean(epoch_real_score))
    mean_fake_score = float(np.mean(epoch_fake_score))

    history_wgan["critic_loss"].append(mean_c_loss)
    history_wgan["generator_loss"].append(mean_g_loss)
    history_wgan["wasserstein_estimate"].append(mean_w_est)
    history_wgan["real_score"].append(mean_real_score)
    history_wgan["fake_score"].append(mean_fake_score)
    if critic_updates % N_CRITIC_WGAN == 0:
        print(
            f"Epoch [{epoch}/{EPOCHS_WGAN}] "
            f"C_loss: {mean_c_loss:.4f} | "
            f"G_loss: {mean_g_loss:.4f} | "
            f"W_est: {mean_w_est:.4f} | "
            f"C(real): {mean_real_score:.4f} | "
            f"C(fake): {mean_fake_score:.4f}"
        )

    if epoch % int(EPOCHS_WGAN / 10) == 0 and epoch != 0:
        wgan_generator.eval()
        with torch.no_grad():
            samples = wgan_generator(fixed_noise)
        show_images(samples, title=f"Conv-WGAN generated images - Epoch {epoch}")

print("WGAN training time:", round(time.time() - start_time, 2), "seconds")


In [ ]:
## 6. Generate images

def generate_images(generator, noise=None, n_images=32):
    generator.eval()
    with torch.no_grad():
        if noise is None:
            noise = torch.randn(n_images, LATENT_DIM, device=device)
        fake_images = generator(noise.to(device))
    return fake_images


wgan_samples = generate_images(wgan_generator, fixed_noise, n_images=32)
show_images(wgan_samples, title="Generated MNIST images with Conv-WGAN", nrow=8, figsize=(8, 4))


In [ ]:
## 7. Training curves and Wasserstein estimate

def plot_history(history, title_prefix="WGAN"):
    epochs = range(1, len(history["critic_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["critic_loss"], label="Critic loss")
    plt.plot(epochs, history["generator_loss"], label="Generator loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} losses")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["wasserstein_estimate"], label="Empirical Wasserstein estimate")
    plt.xlabel("Epoch")
    plt.ylabel("Score gap")
    plt.title(f"{title_prefix}: E[C(real)] - E[C(fake)]")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["real_score"], label="Mean C(real)")
    plt.plot(epochs, history["fake_score"], label="Mean C(fake)")
    plt.xlabel("Epoch")
    plt.ylabel("Critic score")
    plt.title(f"{title_prefix} critic scores")
    plt.legend()
    plt.grid(True)
    plt.show()

    if "gradient_penalty" in history:
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, history["gradient_penalty"], label="Gradient penalty")
        plt.xlabel("Epoch")
        plt.ylabel("GP")
        plt.title(f"{title_prefix} gradient penalty")
        plt.legend()
        plt.grid(True)
        plt.show()

plot_history(history_wgan, title_prefix="WGAN")



### TODO 6 - Analyze WGAN samples and curves

After training, compare generated samples and curves.

Questions:
1. Are the generated digits diverse or do they show mode collapse?
2. Does the critic score for real images stay higher than for fake images?
3. Is weight clipping too restrictive? Look at image quality and score curves.
4. Try changing `CLIP_VALUE` and `N_CRITIC_WGAN`, then record the effect.


## 8. Convolutional WGAN-GP

Weight clipping is simple but crude. It can reduce the capacity of the critic and lead to poor gradients for the generator.

WGAN-GP replaces clipping with a **gradient penalty**:

$$
\lambda \mathbb{E}_{\hat{x}}
\left[
\left(\|\nabla_{\hat{x}} C(\hat{x})\|_2 - 1\right)^2
\right],
$$

where $\hat{x}$ is sampled on straight lines between real and fake images:

$$
\hat{x} = \epsilon x_{\text{real}} + (1-\epsilon)x_{\text{fake}}.
$$

The critic loss becomes:

$$
L_C
=
\mathbb{E}[C(x_{\text{fake}})]
-
\mathbb{E}[C(x_{\text{real}})]
+
\lambda GP.
$$

No weight clipping is used in WGAN-GP.



### TODO 7 - Fill the gradient penalty

Complete the gradient penalty used by WGAN-GP:

$$
GP = \mathbb{E}_{\hat{x}}\left[(\|\nabla_{\hat{x}} C(\hat{x})\|_2 - 1)^2\right].
$$

Questions:
1. Why do we interpolate between real and fake images?
2. Why do we set `requires_grad_(True)` on the interpolated images?
3. Why should the gradient norm be close to 1?
4. Why does WGAN-GP remove the need for weight clipping?


In [ ]:
def gradient_penalty(critic, real_images, fake_images, device):
    batch_size = real_images.size(0)

    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = alpha * real_images + (1 - alpha) * fake_images.detach()
    interpolated.requires_grad_(True)

    interpolated_scores = critic(interpolated)

    gradients = torch.autograd.grad(
        outputs=interpolated_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(interpolated_scores),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.reshape(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    penalty = ((gradient_norm - 1) ** 2).mean()

    return penalty



### TODO 8 - Complete the WGAN-GP critic loss

Fill only the part that changes from WGAN to WGAN-GP: the gradient penalty and the critic loss.

Questions:
1. Which terms are shared with the original WGAN critic loss?
2. Where does the gradient penalty enter the loss?
3. Why is there no weight clipping in WGAN-GP?
4. Compare generated images from WGAN and WGAN-GP. Which one is more stable in your run?


In [ ]:
EPOCHS_GP = 200
N_CRITIC_GP = 5
LAMBDA_GP = 10
LR_G_GP = 1e-4
LR_C_GP = 5e-5

wgan_gp_generator = Generator(LATENT_DIM).to(device)
wgan_gp_critic = Critic().to(device)
wgan_gp_generator.apply(initialize_weights)
wgan_gp_critic.apply(initialize_weights)

optimizer_G = optim.Adam(wgan_gp_generator.parameters(), lr=LR_G_GP, betas=(0.0, 0.9))
optimizer_C = optim.Adam(wgan_gp_critic.parameters(), lr=LR_C_GP, betas=(0.0, 0.9))

fixed_noise_gp = torch.randn(32, LATENT_DIM, device=device)

history_gp = {
    "critic_loss": [],
    "generator_loss": [],
    "wasserstein_estimate": [],
    "gradient_penalty": [],
    "real_score": [],
    "fake_score": []
}

critic_updates = 0
start_time = time.time()

for epoch in range(1, EPOCHS_GP + 1):
    wgan_gp_generator.train()
    wgan_gp_critic.train()

    epoch_c_loss = []
    epoch_g_loss = []
    epoch_w_est = []
    epoch_gp = []
    epoch_real_score = []
    epoch_fake_score = []

    for batch_idx, (real_images, _) in enumerate(train_loader):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # -------------------------
        # 1. Train critic
        # -------------------------
        set_requires_grad(wgan_gp_critic, True)
        set_requires_grad(wgan_gp_generator, False)

        z = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = wgan_gp_generator(z).detach()

        real_scores = wgan_gp_critic(real_images)
        fake_scores = wgan_gp_critic(fake_images)

        # TODO: compute the gradient penalty.
        gp = ...

        # TODO: complete the WGAN-GP critic loss.
        c_loss = (
            ...
            - ...
            + ...
        )

        optimizer_C.zero_grad(set_to_none=True)
        c_loss.backward()
        optimizer_C.step()

        # Important: no weight clipping in WGAN-GP.
        critic_updates += 1

        wasserstein_estimate = real_scores.mean().item() - fake_scores.mean().item()

        epoch_c_loss.append(c_loss.item())
        epoch_w_est.append(wasserstein_estimate)
        epoch_gp.append(gp.item())
        epoch_real_score.append(real_scores.mean().item())
        epoch_fake_score.append(fake_scores.mean().item())

        # -------------------------
        # 2. Train generator every N_CRITIC_GP critic updates
        # -------------------------
        if critic_updates % N_CRITIC_GP == 0:
            set_requires_grad(wgan_gp_critic, False)
            set_requires_grad(wgan_gp_generator, True)

            z = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_images = wgan_gp_generator(z)
            fake_scores_for_g = wgan_gp_critic(fake_images)
            g_loss = generator_loss_wgan(fake_scores_for_g)

            optimizer_G.zero_grad(set_to_none=True)
            g_loss.backward()
            optimizer_G.step()

            epoch_g_loss.append(g_loss.item())

    set_requires_grad(wgan_gp_critic, True)
    set_requires_grad(wgan_gp_generator, True)

    mean_c_loss = float(np.mean(epoch_c_loss))
    mean_g_loss = float(np.mean(epoch_g_loss)) if epoch_g_loss else float("nan")
    mean_w_est = float(np.mean(epoch_w_est))
    mean_gp = float(np.mean(epoch_gp))
    mean_real_score = float(np.mean(epoch_real_score))
    mean_fake_score = float(np.mean(epoch_fake_score))

    history_gp["critic_loss"].append(mean_c_loss)
    history_gp["generator_loss"].append(mean_g_loss)
    history_gp["wasserstein_estimate"].append(mean_w_est)
    history_gp["gradient_penalty"].append(mean_gp)
    history_gp["real_score"].append(mean_real_score)
    history_gp["fake_score"].append(mean_fake_score)

    if critic_updates % N_CRITIC_GP == 0:
        print(
            f"Epoch [{epoch}/{EPOCHS_GP}] "
            f"C_loss: {mean_c_loss:.4f} | "
            f"G_loss: {mean_g_loss:.4f} | "
            f"W_est: {mean_w_est:.4f} | "
            f"GP: {mean_gp:.4f} | "
            f"C(real): {mean_real_score:.4f} | "
            f"C(fake): {mean_fake_score:.4f}"
        )

    if epoch % int(EPOCHS_GP / 10) == 0 and epoch != 0:
        wgan_gp_generator.eval()
        with torch.no_grad():
            samples = wgan_gp_generator(fixed_noise_gp)
        show_images(samples, title=f"Conv-WGAN-GP generated images - Epoch {epoch}")

print("WGAN-GP training time:", round(time.time() - start_time, 2), "seconds")


In [ ]:
wgan_gp_samples = generate_images(wgan_gp_generator, fixed_noise_gp, n_images=32)
show_images(wgan_gp_samples, title="Generated MNIST images with Conv-WGAN-GP", nrow=8, figsize=(8, 4))
plot_history(history_gp, title_prefix="WGAN-GP")


### TODO 9 - Mini-experiment table

Run a small controlled comparison and fill a table like this:

| Run | Method | `N_CRITIC` | Constraint | Final W-estimate | Sample quality | Stability | Comments |
|---|---|---:|---|---:|---|---|---|
| A | WGAN | 5 | weight clipping | | | | |
| B | WGAN | 1 | weight clipping | | | | |
| C | WGAN-GP | 5 | gradient penalty | | | | |
| D | WGAN-GP | 1 | gradient penalty | | | | |

Conclusion to write: explain why the Lipschitz constraint is central to WGAN training and compare weight clipping with gradient penalty.
